Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

    A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
    A function or coroutine to execute.


In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. Parrots are birds known for their ability to mimic human speech. I remember that some parrots can even learn a lot of words and phrases. But why do they do that? Is it just mimicry, or is there a deeper reason?\n\nFirst, I should think about the evolutionary aspect. Maybe parrots talk because it helps them communicate with each other. They might use sounds to signal danger, find mates, or establish territory. But humans are not part of their natural communication environment. So why do they mimic human speech in captivity?\n\nI read somewhere that parrots have a strong social structure. In the wild, they rely on vocalizations to stay in contact with their flock. If they\'re in captivity and kept as pets, they might mimic human speech as a way to bond with their human owners. They might see their owners as part of their social group and try to communicate in t

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. Let\'s format the tool call correctly.\n', 'tool_calls': [{'id': 't0e3mav3d', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 154, 'total_tokens': 246, 'completion_time': 0.145256789, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006233397, 'prompt_tokens_details': None, 'queue_time': 0.12000583, 'total_time': 0.151490186}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run

Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."